In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   MULTIMODAL MEDICAL RAG — SKIN CANCER                      ║
# ║   Complete Pipeline: Phase 1 → 12                           ║
# ║   Single Kaggle Notebook — No external dependencies         ║
# ╚══════════════════════════════════════════════════════════════╝
#
# DATASETS NEEDED (Add Data in Kaggle):
#   1. kmader/skin-cancer-mnist-ham10000
#   2. nodoubttome/skin-cancer9-classesisic
#
# GPU: Enable T4 x2 in Settings → Accelerator


In [ ]:
# CELL 1 — Fixed install order (numpy pinned FIRST)
import subprocess

# Step 1: pin numpy before anything else touches it
subprocess.run(["pip", "install", "numpy==1.26.4", "--force-reinstall", "-q"], check=True)

# Step 2: install everything else
subprocess.run(["pip", "install", "-q",
    "biopython",
    "sentence-transformers",
    "open_clip_torch",
    "qdrant-client",
    "rank-bm25",
    "networkx",
    "beautifulsoup4",
    "lxml",
    "faiss-cpu",
    "google-generativeai",
    "tqdm",
], check=True)

# Step 3: scispaCy LAST (it tries to downgrade numpy — we block it)
subprocess.run(["pip", "install", "-q", "--no-deps", "scispacy"], check=True)
subprocess.run(["pip", "install", "-q",
    "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz"
], check=True)

# Step 4: re-pin numpy after scispaCy (in case it pulled a different version)
subprocess.run(["pip", "install", "numpy==1.26.4", "--force-reinstall", "-q"], check=True)

print("✓ Done. NOW CLICK: Run → Restart & Clear Output, then run Cell 1 again, then Cell 2+")


In [ ]:
# ============================================================
# CELL 1 — Install All Libraries
# ============================================================
!pip install -q \
    biopython \
    sentence-transformers \
    open_clip_torch \
    qdrant-client \
    rank-bm25 \
    scispacy \
    networkx \
    beautifulsoup4 \
    lxml \
    faiss-cpu \
    ragas \
    deepeval \
    google-generativeai \
    tqdm

!pip install -q groq
!pip install -q \
    https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

print("All libraries installed.")


In [ ]:
# ============================================================
# CELL 2 — Imports & Global Config
# ============================================================
import os, re, json, time, shutil, warnings, itertools
import numpy as np
import pandas as pd
import networkx as nx
import torch
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from collections import defaultdict, Counter
from typing import List, Dict, Optional

# Bio imports
from Bio import Entrez, Medline

# Vision
from PIL import Image
import open_clip

# NLP
import spacy
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from bs4 import BeautifulSoup
from xml.etree import ElementTree as ET

# Vector DB
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)

# LLM
import google.generativeai as genai

warnings.filterwarnings("ignore")

# ── Device ────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── Directories ───────────────────────────────────────────────
BASE_DIR  = Path("/kaggle/working/data")
DIRS = [
    BASE_DIR / "pubmed",
    BASE_DIR / "pmc_fulltext",
    BASE_DIR / "images/metadata",
    BASE_DIR / "chunks",
    BASE_DIR / "embeddings",
    BASE_DIR / "qdrant_storage",
    BASE_DIR / "knowledge_graph",
    BASE_DIR / "evaluation",
    BASE_DIR / "logs",
]
for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)

# ── Kaggle dataset paths ───────────────────────────────────────
ISIC_DIR = Path("/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration")

# HAM10000 dataset
HAM_DIR = Path(
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
)

print("ISIC_DIR exists :", ISIC_DIR.exists())
print("HAM_DIR exists  :", HAM_DIR.exists())


print("Imports done. Directories ready.")


In [ ]:
# ============================================================
# CELL 3 — PHASE 1A: PubMed Data Collection
# ============================================================
print("\n" + "="*60)
print("PHASE 1 — DATA COLLECTION")
print("="*60)

Entrez.email = "your_email@gmail.com"

SKIN_CANCER_QUERY = """
(
    melanoma OR "skin cancer" OR "basal cell carcinoma"
    OR "squamous cell carcinoma" OR "actinic keratosis"
    OR "melanocytic nevus" OR "skin lesion"
)
AND (
    dermoscopy OR dermatoscopy OR "skin lesion classification"
    OR diagnosis OR segmentation OR classification
)
AND (
    "deep learning" OR CNN OR transformer OR "vision transformer"
    OR "machine learning" OR "artificial intelligence"
)
"""

# ── STEP 1: Define all functions ──────────────────────────────
def search_pubmed(query, max_results=500):
    all_ids = []
    for retstart in range(0, max_results, 100):
        try:
            handle = Entrez.esearch(db="pubmed", term=query,
                                    retmax=100, retstart=retstart,
                                    sort="relevance")
            record = Entrez.read(handle)
            ids    = record["IdList"]
            all_ids.extend(ids)
            time.sleep(0.4)
            if len(ids) < 100:
                break
        except Exception as e:
            print(f"Search error at {retstart}: {e}")
            time.sleep(2)
    return all_ids[:max_results]


def fetch_abstracts(paper_ids, batch_size=50):
    records_list = []
    for i in tqdm(range(0, len(paper_ids), batch_size),
                  desc="Fetching abstracts"):
        batch = paper_ids[i:i+batch_size]
        try:
            handle = Entrez.efetch(db="pubmed", id=",".join(batch),
                                   rettype="medline", retmode="text")
            for r in Medline.parse(handle):
                records_list.append({
                    "pmid":             r.get("PMID", ""),
                    "title":            r.get("TI",   ""),
                    "abstract":         r.get("AB",   ""),
                    "journal":          r.get("JT",   ""),
                    "publication_date": r.get("DP",   ""),
                    "authors":          ", ".join(r.get("AU", [])[:10]),
                    "mesh_terms":       "; ".join(r.get("MH", [])),
                    "keywords":         "; ".join(
                        r.get("OT", []) if isinstance(r.get("OT",[]), list)
                        else [str(r.get("OT",""))]
                    ),
                    "source": "PubMed"
                })
            time.sleep(0.4)
        except Exception as e:
            print(f"Fetch error batch {i}: {e}")
            time.sleep(2)
    return pd.DataFrame(records_list)


def get_pmc_ids(paper_ids, batch_size=150, email="your_email@gmail.com"):
    import requests
    BASE_URL    = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"
    pmc_records = []
    for i in tqdm(range(0, len(paper_ids), batch_size),
                  desc="PMC ID lookup"):
        batch = paper_ids[i:i+batch_size]
        for attempt in range(3):
            try:
                r = requests.get(BASE_URL, params={
                    "ids": ",".join(batch), "idtype": "pmid",
                    "format": "xml", "tool": "skin_rag",
                    "email": email
                }, timeout=30)
                if r.status_code != 200:
                    raise ValueError(f"HTTP {r.status_code}")
                root = ET.fromstring(r.content)
                for rec in root.findall(".//record"):
                    pmid  = rec.get("pmid",  "")
                    pmcid = rec.get("pmcid", "")
                    if pmcid:
                        pmc_records.append({"pmid":  pmid,
                                            "pmcid": pmcid,
                                            "doi":   rec.get("doi","")})
                time.sleep(0.5)
                break
            except Exception:
                time.sleep(2)
    return pd.DataFrame(pmc_records) if pmc_records else pd.DataFrame(
        columns=["pmid","pmcid","doi"])


def download_fulltext(pmcid, save_dir):
    import requests
    # Strategy 1: Europe PMC
    try:
        r = requests.get(
            f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML",
            timeout=30, headers={"Accept": "application/xml"})
        if (r.status_code == 200 and len(r.text) > 5000 and
                ("<?xml" in r.text[:100] or "<!DOCTYPE article" in r.text[:100])):
            path = save_dir / f"{pmcid}_europepmc.xml"
            path.write_text(r.text, encoding="utf-8")
            return str(path), "europepmc"
    except: pass

    # Strategy 2: BioC API
    try:
        numeric = pmcid.replace("PMC", "")
        r = requests.get(
            f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/"
            f"pmcoa.cgi/BioC_xml/{numeric}/unicode", timeout=30)
        if r.status_code == 200 and "<collection>" in r.text and len(r.text) > 3000:
            path = save_dir / f"{pmcid}_bioc.xml"
            path.write_text(r.text, encoding="utf-8")
            return str(path), "bioc"
    except: pass

    # Strategy 3: PMC OAI
    try:
        r = requests.get(
            f"https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi"
            f"?verb=GetRecord"
            f"&identifier=oai:pubmedcentral.nih.gov:{pmcid.replace('PMC','')}"
            f"&metadataPrefix=pmc",
            timeout=30)
        if r.status_code == 200 and "<article" in r.text and len(r.text) > 5000:
            path = save_dir / f"{pmcid}_oai.xml"
            path.write_text(r.text, encoding="utf-8")
            return str(path), "oai"
    except: pass

    return None, "failed"


def chunk_text_p1(text, max_words=250, overlap=40):
    words = text.split()
    if not words: return []
    if len(words) <= max_words: return [text]
    step, chunks, start = max_words - overlap, [], 0
    while start < len(words):
        end   = min(start + max_words, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk.strip())
        start += step
    return chunks


# ── STEP 2: Run data collection ───────────────────────────────
print("Searching PubMed...")
paper_ids = search_pubmed(SKIN_CANCER_QUERY, max_results=500)
print(f"Found {len(paper_ids)} papers")

print("Fetching abstracts...")
df_abstracts = fetch_abstracts(paper_ids)
df_abstracts.to_csv(BASE_DIR/"pubmed"/"abstracts.csv", index=False)
print(f"Abstracts saved: {len(df_abstracts)}")

print("Getting PMC IDs...")
df_pmc = get_pmc_ids(paper_ids)
df_pmc.to_csv(BASE_DIR/"pubmed"/"pmc_ids.csv", index=False)
print(f"PMC-linked: {len(df_pmc)}")

print("Downloading full text (up to 150 articles)...")
downloaded, failed = [], []
for _, row in tqdm(df_pmc.head(150).iterrows(), total=150, desc="Full text"):
    path, method = download_fulltext(row["pmcid"], BASE_DIR/"pmc_fulltext")
    if path:
        downloaded.append({"pmid":   row["pmid"],
                           "pmcid":  row["pmcid"],
                           "method": method,
                           "path":   path})
    else:
        failed.append(row["pmcid"])
    time.sleep(0.4)

df_downloaded = pd.DataFrame(downloaded)
df_downloaded.to_csv(BASE_DIR/"pubmed"/"downloaded_manifest.csv", index=False)
print(f"Downloaded: {len(downloaded)} | Failed: {len(failed)}")


# ── STEP 3: Build RAG chunks ──────────────────────────────────
print("\nBuilding RAG chunks...")
all_chunks = []

for _, row in df_downloaded.iterrows():
    pmcid = row["pmcid"]
    path  = Path(row["path"])
    if not path.exists():
        continue
    raw  = path.read_text(encoding="utf-8", errors="ignore")
    text = re.sub(r"<[^>]+>", " ", raw)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) > 300:
        for i, chunk in enumerate(chunk_text_p1(text)):
            all_chunks.append({
                "chunk_id":    f"{pmcid}_c{i:03d}",
                "pmcid":       pmcid,
                "pmid":        str(row.get("pmid", "")),
                "source_type": "PMC XML",
                "section":     "Full Text",
                "text":        chunk,
                "word_count":  len(chunk.split())
            })

downloaded_pmids = set(df_downloaded["pmid"].astype(str))
for _, row in df_abstracts.iterrows():
    pmid = str(row["pmid"])
    if pmid in downloaded_pmids:
        continue
    ab = str(row.get("abstract","")).strip()
    if len(ab) < 50:
        continue
    for i, chunk in enumerate(chunk_text_p1(ab, max_words=200)):
        all_chunks.append({
            "chunk_id":    f"{pmid}_abs_c{i:03d}",
            "pmcid":       "",
            "pmid":        pmid,
            "source_type": "PubMed Abstract",
            "section":     "Abstract",
            "text":        chunk,
            "word_count":  len(chunk.split()),
            "title":       str(row.get("title",  "")),
            "journal":     str(row.get("journal","")),
        })

df_chunks = pd.DataFrame(all_chunks)
df_chunks  = df_chunks[
    df_chunks["text"].str.strip().str.len() > 50
].reset_index(drop=True)
df_chunks.to_csv(BASE_DIR/"chunks"/"text_chunks.csv", index=False)


# ── STEP 4: Summary ───────────────────────────────────────────
rag_chunks_fulltext      = int((df_chunks["source_type"]=="PMC XML").sum())
rag_chunks_abstract_only = int((df_chunks["source_type"]=="PubMed Abstract").sum())

print(f"\n{'='*60}")
print("PHASE 1A SUMMARY")
print(f"{'='*60}")
for k, v in {
    "pubmed_papers_found":      len(paper_ids),
    "abstracts_saved":          len(df_abstracts),
    "pmc_links_found":          len(df_pmc),
    "pmc_xml_downloaded":       len(df_downloaded),
    "pmc_xml_paywalled":        len(failed),
    "rag_chunks_total":         len(df_chunks),
    "rag_chunks_fulltext":      rag_chunks_fulltext,
    "rag_chunks_abstract_only": rag_chunks_abstract_only,
}.items():
    print(f"  {k:<35}: {v}")
    

In [ ]:
# ============================================================
# CELL 4 — PHASE 1B: Image Manifest (HAM10000 + ISIC)
# ============================================================


all_records = []

# ── HAM10000 (correct path) ───────────────────────────────────────────────────
HAM_DIR = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")

ham_meta_path = HAM_DIR / "HAM10000_metadata.csv"
if ham_meta_path.exists():
    ham_meta = pd.read_csv(ham_meta_path)
    dx_map   = {
        "akiec": "actinic keratosis",   "bcc": "basal cell carcinoma",
        "bkl":   "benign keratosis",    "df":  "dermatofibroma",
        "mel":   "melanoma",            "nv":  "nevus",
        "vasc":  "vascular lesion"
    }
    ham_meta["label"] = ham_meta["dx"].map(dx_map)
    label_map = dict(zip(ham_meta["image_id"], ham_meta["label"]))

    for folder in [HAM_DIR/"HAM10000_images_part_1",
                   HAM_DIR/"HAM10000_images_part_2"]:
        if not folder.exists():
            print(f"Missing: {folder}")
            continue
        for img in folder.glob("*.jpg"):
            all_records.append({
                "image_id":   img.stem,
                "image_path": str(img),
                "dataset":    "HAM10000",
                "split":      "unknown",
                "label":      label_map.get(img.stem, "unknown")
            })
    print(f"HAM10000: {len(all_records)} images")
else:
    print(f"HAM10000 metadata not found at {ham_meta_path}")

# ── ISIC — auto-discover correct path ────────────────────────────────────────
ISIC_ROOT = None
for candidate in Path("/kaggle/input").rglob("Train"):
    if candidate.is_dir() and "isic" in str(candidate).lower():
        ISIC_ROOT = candidate.parent
        print(f"ISIC found: {ISIC_ROOT}")
        break

# Broader search if not found
if ISIC_ROOT is None:
    for candidate in Path("/kaggle/input").rglob("Train"):
        if candidate.is_dir():
            # Check if it has image subfolders
            subdirs = [d for d in candidate.iterdir() if d.is_dir()]
            if len(subdirs) > 2:
                ISIC_ROOT = candidate.parent
                print(f"ISIC found (broad search): {ISIC_ROOT}")
                break

isic_count = 0
if ISIC_ROOT:
    for split in ["Train", "Test"]:
        split_path = ISIC_ROOT / split
        if not split_path.exists():
            continue
        for cls in split_path.iterdir():
            if not cls.is_dir():
                continue
            for img in cls.glob("*.jpg"):
                all_records.append({
                    "image_id":   img.stem,
                    "image_path": str(img),
                    "dataset":    "ISIC",
                    "split":      split,
                    "label":      cls.name.lower().strip()
                })
                isic_count += 1
    print(f"ISIC: {isic_count} images")
else:
    print("ISIC not mounted yet — save & restart notebook to load it")

# ── Build manifest ────────────────────────────────────────────────────────────
if not all_records:
    print("WARNING: No images found.")
    df_images = pd.DataFrame(columns=["image_id","image_path","dataset",
                                       "split","label","image_text_description"])
else:
    df_images = pd.DataFrame(all_records).drop_duplicates(subset=["image_path"])
    df_images["image_text_description"] = (
        "Dermoscopy image from " + df_images["dataset"] + " dataset. " +
        "Diagnostic label: "     + df_images["label"]   + ". " +
        "Split: "                + df_images["split"]   + "."
    )

df_images.to_csv(BASE_DIR/"images/metadata"/"image_manifest.csv", index=False)
print(f"\nTotal images indexed : {len(df_images)}")
if len(df_images) > 0:
    print(df_images["dataset"].value_counts())
    print(df_images["label"].value_counts().head(10))
    

In [ ]:
# CELL 5 — Fixed: proper JATS/BioC parser matching old Phase 2 quality

print("\n" + "="*60)
print("PHASE 2 — DOCUMENT PROCESSING")
print("="*60)

from bs4 import BeautifulSoup
import re

SKIP_TAGS = {"xref","ref","ref-list","table","fig",
             "supplementary-material","media","ext-link"}

def clean_text(text):
    if not text: return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\[\s*[\d,\s\-]+\s*\]", "", text)
    text = re.sub(r"\(\s*(Fig|Figure|Table)\.?\s*\d+\s*\)",
                  "", text, flags=re.IGNORECASE)
    return text.strip()


def parse_xml_proper(xml_path: Path) -> dict:
    """Full JATS/BioC parser — matches old Phase 2 quality."""
    raw = xml_path.read_text(encoding="utf-8", errors="ignore")

    # Skip HTML files (old broken downloads)
    if raw.lstrip().startswith("<!DOCTYPE html") or "<html" in raw[:200]:
        return {"title":"","abstract":"","sections":[],"figures":[]}

    soup = BeautifulSoup(raw, "lxml-xml")

    # ── BioC format ───────────────────────────────────────────
    if soup.find("collection"):
        passages = []
        for p in soup.find_all("passage"):
            infon = ""
            for i in p.find_all("infon"):
                if i.get("key") == "type":
                    infon = (i.text or "").lower()
            if any(x in infon for x in ["table","fig","ref","caption"]):
                continue
            t = p.find("text")
            if t and t.text and len(t.text.strip()) > 20:
                passages.append({"section_title": infon or "Body",
                                 "text": clean_text(t.text)})
        # group by section
        sections = {}
        for p in passages:
            k = p["section_title"]
            sections.setdefault(k, []).append(p["text"])
        return {
            "title":    "",
            "abstract": "",
            "sections": [{"section_title": k, "text": " ".join(v)}
                          for k,v in sections.items()],
            "figures":  []
        }

    # ── JATS format ───────────────────────────────────────────
    title = ""
    t_tag = soup.find("article-title")
    if t_tag: title = clean_text(t_tag.get_text(" "))

    abstract = ""
    a_tag = soup.find("abstract")
    if a_tag: abstract = clean_text(a_tag.get_text(" "))

    sections = []
    body = soup.find("body")
    if body:
        for sec in body.find_all("sec", recursive=True):
            h       = sec.find("title", recursive=False)
            heading = clean_text(h.get_text(" ")) if h else "Section"
            paras   = []
            for p in sec.find_all("p", recursive=False):
                txt = clean_text(p.get_text(" "))
                if len(txt) > 30:
                    paras.append(txt)
            if paras:
                sections.append({"section_title": heading,
                                 "text": " ".join(paras)})

    figures = []
    for fig in soup.find_all("fig"):
        label   = fig.find("label")
        caption = fig.find("caption")
        cap_txt = clean_text(caption.get_text(" ")) if caption else ""
        lbl_txt = clean_text(label.get_text(" "))   if label   else ""
        if cap_txt and len(cap_txt) > 30:
            figures.append({"figure_label": lbl_txt, "caption": cap_txt})

    return {"title":title,"abstract":abstract,
            "sections":sections,"figures":figures}


def chunk_text(text, max_words=250, overlap=40):
    words = text.split()
    if not words: return []
    if len(words) <= max_words: return [text]
    step, chunks, start = max_words - overlap, [], 0
    while start < len(words):
        end   = min(start + max_words, len(words))
        chunk = " ".join(words[start:end])
        if end < len(words):
            boundary = int(len(chunk) * 0.80)
            lp = max(chunk.rfind(". ", boundary),
                     chunk.rfind(".\n", boundary))
            if lp > 0: chunk = chunk[:lp+1]
        chunks.append(chunk.strip())
        start += step
    return chunks


# ── Parse all XML files ───────────────────────────────────────
xml_files = list((BASE_DIR/"pmc_fulltext").glob("*.xml"))
print(f"Parsing {len(xml_files)} XML files...")

all_chunks        = []
fulltext_pmcids   = set()

for xf in tqdm(xml_files, desc="Parsing XML"):
    pmcid  = xf.stem.split("_")[0]
    parsed = parse_xml_proper(xf)
    pmid_match = df_downloaded.loc[
        df_downloaded["pmcid"] == pmcid, "pmid"
    ].values
    pmid = str(pmid_match[0]) if len(pmid_match) > 0 else ""

    has_content = (len(parsed.get("abstract","")) > 50 or
                   len(parsed.get("sections",[])) > 0)
    if not has_content:
        continue

    fulltext_pmcids.add(pmcid)

    # Abstract chunk
    if parsed["abstract"]:
        for i, chunk in enumerate(chunk_text(parsed["abstract"], max_words=200)):
            all_chunks.append({
                "chunk_id":    f"{pmcid}_abstract_{i}",
                "pmcid":       pmcid, "pmid": pmid,
                "source_type": "PMC XML",
                "section":     "Abstract",
                "text":        chunk,
                "word_count":  len(chunk.split()),
                "title":       parsed["title"],
            })

    # Section chunks
    for sec in parsed["sections"]:
        heading = sec["section_title"]
        text    = sec["text"]
        if not text or len(text) < 30: continue
        for i, chunk in enumerate(chunk_text(text, max_words=250)):
            all_chunks.append({
                "chunk_id":    f"{pmcid}_{re.sub(r'[^a-zA-Z0-9]','_',heading)}_{i}",
                "pmcid":       pmcid, "pmid": pmid,
                "source_type": "PMC XML",
                "section":     heading,
                "text":        chunk,
                "word_count":  len(chunk.split()),
                "title":       parsed["title"],
            })

    # Figure caption chunks
    for fi, fig in enumerate(parsed["figures"]):
        cap  = fig["caption"]
        lbl  = fig.get("figure_label", f"Figure {fi+1}")
        text = clean_text(f"{lbl}. {cap}")
        if len(text) > 30:
            all_chunks.append({
                "chunk_id":    f"{pmcid}_fig_{fi}",
                "pmcid":       pmcid, "pmid": pmid,
                "source_type": "PMC XML",
                "section":     "Figure Caption",
                "text":        text,
                "word_count":  len(text.split()),
                "title":       parsed["title"],
            })

# ── Abstract-only for paywalled ───────────────────────────────
downloaded_pmids = set(df_downloaded["pmid"].astype(str))
for _, row in df_abstracts.iterrows():
    pmid = str(row["pmid"])
    if pmid in downloaded_pmids: continue
    ab = str(row.get("abstract","")).strip()
    if len(ab) < 50: continue
    for i, chunk in enumerate(chunk_text(ab, max_words=200)):
        all_chunks.append({
            "chunk_id":    f"{pmid}_abs_c{i:03d}",
            "pmcid":       "", "pmid": pmid,
            "source_type": "PubMed Abstract",
            "section":     "Abstract",
            "text":        chunk,
            "word_count":  len(chunk.split()),
            "title":       str(row.get("title","")),
            "journal":     str(row.get("journal","")),
        })

df_chunks = pd.DataFrame(all_chunks)
df_chunks  = df_chunks[df_chunks["text"].str.strip().str.len() > 50].reset_index(drop=True)

# Fill missing columns
for col in ["journal","title","section","pmid","pmcid"]:
    if col not in df_chunks.columns:
        df_chunks[col] = ""
    df_chunks[col] = df_chunks[col].fillna("")

df_chunks.to_csv(BASE_DIR/"chunks"/"text_chunks.csv", index=False)

print(f"\nChunk Summary:")
print(f"  Total chunks   : {len(df_chunks)}")
print(df_chunks["source_type"].value_counts())
print(f"\nSection distribution (top 15):")
print(df_chunks["section"].value_counts().head(15))
print(f"  Avg words/chunk: {df_chunks['word_count'].mean():.0f}")


In [ ]:
# ============================================================
# CELL 6 — PHASE 3A: Text Embeddings (BGE-large)
# ============================================================
print("\n" + "="*60)
print("PHASE 3 — TEXT EMBEDDINGS")
print("="*60)

print("Loading BGE-large-en-v1.5...")
text_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=DEVICE)
TEXT_DIM   = text_model.get_sentence_embedding_dimension()
print(f"Text embedding dim: {TEXT_DIM}")

BATCH_SIZE = 64

print(f"Embedding {len(df_chunks)} chunks...")
text_embeddings = text_model.encode(
    df_chunks["text"].tolist(),
    normalize_embeddings=True,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
).astype(np.float32)

np.save(BASE_DIR/"embeddings"/"text_embeddings.npy", text_embeddings)
print(f"Text embeddings saved: {text_embeddings.shape}")

del text_model
if DEVICE=="cuda": torch.cuda.empty_cache()

# PHASE 3A SUMMARY — add at end of Cell 6
print(f"\n{'='*60}")
print("PHASE 3A — TEXT EMBEDDING SUMMARY")
print(f"{'='*60}")
print(f"Model            : BAAI/bge-large-en-v1.5")
print(f"Embedding dim    : {TEXT_DIM}")
print(f"Chunks embedded  : {text_embeddings.shape[0]}")
print(f"Embedding shape  : {text_embeddings.shape}")
print(f"Embedding dtype  : {text_embeddings.dtype}")
print(f"Saved to         : {BASE_DIR}/embeddings/text_embeddings.npy")

# Sanity check — L2 norms should all be ~1.0 (normalized)
norms = np.linalg.norm(text_embeddings, axis=1)
print(f"Norm range       : {norms.min():.4f} – {norms.max():.4f}  (should be ~1.0)")
print(f"✓ Text embeddings ready — proceed to Cell 7 (image embeddings)")


In [ ]:
# ============================================================
# CELL 7 — PHASE 3B: Image Embeddings (BiomedCLIP)
# ============================================================
print("\n" + "="*60)
print("PHASE 3 — IMAGE EMBEDDINGS")
print("="*60)

print("Loading BiomedCLIP...")
try:
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    )
    clip_tokenizer = open_clip.get_tokenizer(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    )
    IMAGE_MODEL_NAME = "BiomedCLIP"
except Exception as e:
    print(f"BiomedCLIP failed ({e}) — using CLIP ViT-B/32")
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai"
    )
    clip_tokenizer   = open_clip.get_tokenizer("ViT-B-32")
    IMAGE_MODEL_NAME = "CLIP-ViT-B/32"

clip_model = clip_model.to(DEVICE)
clip_model.eval()

def get_image_dim(model):
    if hasattr(model, "visual") and hasattr(model.visual, "output_dim"):
        return model.visual.output_dim
    dummy = torch.zeros(1, 3, 224, 224).to(DEVICE)
    with torch.no_grad():
        return model.encode_image(dummy).shape[-1]

IMAGE_DIM = get_image_dim(clip_model)
print(f"Image embedding dim: {IMAGE_DIM}")

IMG_BATCH      = 32
FALLBACK_BATCH = 64

def embed_images(df, model, preprocess, img_batch=IMG_BATCH, fb_batch=FALLBACK_BATCH):
    all_embs            = []
    img_batch_data      = []
    img_idx             = []

    def flush():
        if not img_batch_data:
            return
        tensor = torch.stack(img_batch_data).to(DEVICE)
        with torch.no_grad():
            embs = model.encode_image(tensor)
            embs = embs / embs.norm(dim=-1, keepdim=True)
        for idx, emb in zip(img_idx, embs.cpu().numpy()):
            all_embs.append((idx, emb.astype(np.float32), "image"))
        img_batch_data.clear()
        img_idx.clear()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Loading images"):
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            img_batch_data.append(preprocess(img))
            img_idx.append(i)
            if len(img_batch_data) >= img_batch:
                flush()
        except:
            all_embs.append((i, None, "text_fallback"))
    flush()

    fallback_idx = [i for i, e, s in all_embs if e is None]
    if fallback_idx:
        print(f"  Text fallback for {len(fallback_idx)} images...")
        texts  = df.loc[fallback_idx, "image_text_description"].tolist()
        f_embs = []
        for b in tqdm(range(0, len(texts), fb_batch), desc="  Fallback"):
            tokens = clip_tokenizer(texts[b:b + fb_batch]).to(DEVICE)
            with torch.no_grad():
                e = model.encode_text(tokens)
                e = e / e.norm(dim=-1, keepdim=True)
            f_embs.append(e.cpu().numpy().astype(np.float32))
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        f_embs_np = np.vstack(f_embs)
        f_map     = dict(zip(fallback_idx, f_embs_np))
        all_embs  = [(i, f_map[i] if e is None else e, s)
                     for i, e, s in all_embs]

    all_embs.sort(key=lambda x: x[0])
    return (np.vstack([e for _, e, _ in all_embs]),
            [s for _, _, s in all_embs])


print(f"Embedding {len(df_images)} images...")
image_embeddings, img_sources = embed_images(df_images, clip_model, clip_preprocess)
df_images["embed_source"]     = img_sources
np.save(BASE_DIR / "embeddings" / "image_embeddings.npy", image_embeddings)
print(f"Image embeddings saved: {image_embeddings.shape}")

# ── Keep CLIP model alive for image retrieval in Cell 11 ──────
# Do NOT delete clip_model here — retrieve_images() needs it
mm_model     = clip_model       # alias used by retrieve_images()
mm_tokenizer = clip_tokenizer   # alias used by retrieve_images()
print(f"mm_model ready : {IMAGE_MODEL_NAME}, dim={IMAGE_DIM}")

# ── Phase 3B Summary ──────────────────────────────────────────
print(f"\n{'='*60}")
print("PHASE 3B — IMAGE EMBEDDING SUMMARY")
print(f"{'='*60}")
print(f"Model            : {IMAGE_MODEL_NAME}")
print(f"Embedding dim    : {IMAGE_DIM}")
print(f"Images embedded  : {image_embeddings.shape[0]}")
print(f"Embedding shape  : {image_embeddings.shape}")
print(f"Embedding dtype  : {image_embeddings.dtype}")
print(f"Saved to         : {BASE_DIR}/embeddings/image_embeddings.npy")

source_counts = pd.Series(img_sources).value_counts()
print(f"\nEmbed source breakdown:")
for src, count in source_counts.items():
    pct = count / len(img_sources) * 100
    print(f"  {src:<20}: {count:>6}  ({pct:.1f}%)")

norms = np.linalg.norm(image_embeddings, axis=1)
print(f"\nNorm range       : {norms.min():.4f} – {norms.max():.4f}  (should be ~1.0)")

print(f"\nImages by dataset:")
print(df_images["dataset"].value_counts().to_string())
print(f"\nImages by label (top 10):")
print(df_images["label"].value_counts().head(10).to_string())

print(f"\n✓ Image embeddings ready — proceed to Cell 8 (Qdrant)")


In [ ]:
# CELL 8 — PHASE 4: Qdrant Vector Database

print("\n" + "="*60)
print("PHASE 4 — VECTOR DATABASE")
print("="*60)

QDRANT_PATH      = str(BASE_DIR/"qdrant_storage")
TEXT_COLLECTION  = "skin_cancer_text"
IMAGE_COLLECTION = "skin_cancer_images"

# ── Close any existing Qdrant instance first ──────────────────
try:
    qdrant.close()
    print("Previous Qdrant instance closed.")
except:
    pass

# ── Also release file lock by force if needed ─────────────────
import gc
gc.collect()

qdrant = QdrantClient(path=QDRANT_PATH)

for name, dim in [(TEXT_COLLECTION, TEXT_DIM),
                  (IMAGE_COLLECTION, IMAGE_DIM)]:
    existing = [c.name for c in qdrant.get_collections().collections]
    if name in existing:
        qdrant.delete_collection(name)
    qdrant.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
    )

UPLOAD_BATCH = 256

def upload_points(collection, df, embeddings, payload_fn):
    for start in tqdm(range(0, len(df), UPLOAD_BATCH),
                      desc=f"Uploading {collection}"):
        end    = min(start + UPLOAD_BATCH, len(df))
        points = [
            PointStruct(
                id      = start + i,
                vector  = embeddings[start + i].tolist(),
                payload = payload_fn(df.iloc[start + i])
            )
            for i in range(end - start)
        ]
        qdrant.upsert(collection_name=collection, points=points)

upload_points(TEXT_COLLECTION, df_chunks, text_embeddings, lambda r: {
    "chunk_id":    str(r.get("chunk_id",    "")),
    "pmid":        str(r.get("pmid",        "")),
    "pmcid":       str(r.get("pmcid",       "")),
    "title":       str(r.get("title",       ""))[:500],
    "section":     str(r.get("section",     "")),
    "source_type": str(r.get("source_type", "")),
    "text":        str(r.get("text",        ""))[:2000],
})

upload_points(IMAGE_COLLECTION, df_images, image_embeddings, lambda r: {
    "image_id":               str(r.get("image_id",               "")),
    "image_path":             str(r.get("image_path",             "")),
    "label":                  str(r.get("label",                  "")),
    "dataset":                str(r.get("dataset",                "")),
    "split":                  str(r.get("split",                  "")),
    "image_text_description": str(r.get("image_text_description", ""))[:500],
    "embed_source":           str(r.get("embed_source",           "")),
})

for name in [TEXT_COLLECTION, IMAGE_COLLECTION]:
    info  = qdrant.get_collection(name)
    count = info.points_count if hasattr(info, "points_count") else info.vectors_count
    print(f"  {name}: {count} vectors")

print("Qdrant ready.")


In [ ]:
# ============================================================
# CELL 9 — PHASE 5: Knowledge Graph
# ============================================================
print("\n" + "="*60)
print("PHASE 5 — KNOWLEDGE GRAPH")
print("="*60)

try:
    nlp = spacy.load("en_ner_bc5cdr_md")
    print("scispaCy loaded.")
except:
    nlp = spacy.load("en_core_web_sm")
    print("Fallback: en_core_web_sm")

ENTITY_LEXICON = {
    "DISEASE_SKIN": ["melanoma","basal cell carcinoma","squamous cell carcinoma",
                     "actinic keratosis","nevus","skin cancer","skin lesion",
                     "melanocytic nevus","dermatofibroma","bcc","scc",
                     "benign keratosis","vascular lesion"],
    "TECHNIQUE":    ["dermoscopy","dermatoscopy","biopsy","whole slide imaging",
                     "reflectance confocal microscopy"],
    "MODEL":        ["cnn","convolutional neural network","resnet","vgg","inception",
                     "efficientnet","densenet","vision transformer","vit","swin",
                     "unet","u-net","transformer","yolo","deep learning",
                     "machine learning","artificial intelligence","random forest","svm"],
    "METRIC":       ["accuracy","sensitivity","specificity","auc","roc",
                     "f1","f1 score","precision","recall","iou","dice coefficient"],
    "DATASET":      ["ham10000","isic","isic 2018","isic 2019","ph2","derm7pt"],
    "BODY_SITE":    ["skin","dermis","epidermis","melanocyte","keratinocyte"],
}

LEXICON_MAP = {term.lower(): etype
               for etype, terms in ENTITY_LEXICON.items()
               for term in terms}

def get_entities(text):
    found = {}
    text_lower = text.lower()
    for phrase, etype in LEXICON_MAP.items():
        if phrase in text_lower:
            found[phrase] = {"text":phrase,"label":etype}
    try:
        doc = nlp(text[:10000])
        for ent in doc.ents:
            key = ent.text.lower().strip()
            if len(key)>2 and key not in found:
                found[key] = {"text":key,"label":ent.label_}
    except: pass
    return list(found.values())

RELATION_PATTERNS = [
    (r"([\w\s\-]+?)\s+(?:detects?|classif(?:ies|y)|diagnos(?:es|e)|segments?)\s+([\w\s\-]+)","DETECTS"),
    (r"([\w\s\-]+?)\s+(?:achieved?|obtained?|reached?)\s+(?:an?\s+)?(accuracy|auc|sensitivity|specificity|f1)\s+of\s+([\d\.]+\s*%?)","ACHIEVES"),
    (r"([\w\s\-]+?)\s+(?:trained?|evaluated?|tested?)\s+(?:on|using)\s+(?:the\s+)?(ham10000|isic[\s\w]*|ph2)","TRAINED_ON"),
    (r"([\w\s\-]+?)\s+(?:improves?|outperforms?|enhances?)\s+([\w\s\-]+)","IMPROVES"),
]

print("Building knowledge graph...")
G            = nx.DiGraph()
entity_freq  = defaultdict(int)
entity_types = {}
edge_weights = defaultdict(int)
edge_evidence= defaultdict(list)

for _, row in tqdm(df_chunks.iterrows(), total=len(df_chunks), desc="Extracting entities"):
    text    = str(row["text"])
    cid     = str(row["chunk_id"])
    entities= get_entities(text)

    for e in entities:
        k = e["text"]
        entity_freq[k]  += 1
        entity_types[k]  = e["label"]

    # Pattern relations
    for pattern, rel_type in RELATION_PATTERNS:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            groups = [g.strip().lower() for g in match.groups() if g]
            if len(groups)>=2:
                subj, obj = groups[0][-60:], groups[1][:60]
                if len(subj)>2 and len(obj)>2 and subj!=obj:
                    key = (subj, rel_type, obj)
                    edge_weights[key]  += 1
                    edge_evidence[key].append(cid)

    # Co-occurrence
    sentences = re.split(r"(?<=[.!?])\s+", text)
    for sent in sentences:
        sl    = sent.lower()
        found = [e for e in entities if e["text"] in sl]
        for e1,e2 in itertools.combinations(found,2):
            if e1["label"]==e2["label"]=="METRIC": continue
            key = (e1["text"],"CO_OCCURS",e2["text"])
            edge_weights[key]  += 1
            edge_evidence[key].append(cid)

# Build graph
for ent, freq in entity_freq.items():
    G.add_node(ent, label=ent,
               entity_type=entity_types.get(ent,"UNKNOWN"),
               frequency=freq)

for (subj,rel,obj), weight in edge_weights.items():
    if not subj or not obj or subj==obj: continue
    if subj not in G:
        G.add_node(subj,label=subj,entity_type="UNKNOWN",frequency=1)
    if obj not in G:
        G.add_node(obj, label=obj, entity_type="UNKNOWN",frequency=1)
    evidence = "; ".join(list(set(edge_evidence[(subj,rel,obj)]))[:5])
    G.add_edge(subj,obj,relation=rel,weight=weight,evidence=evidence)

nx.write_graphml(G, str(BASE_DIR/"knowledge_graph"/"skin_cancer_kg.graphml"))
print(f"KG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")




In [ ]:
# PHASE 4 + 5 SUMMARY — add after Cell 9 (KG construction)

print(f"\n{'='*60}")
print("PHASE 4 — VECTOR DATABASE SUMMARY")
print(f"{'='*60}")

text_info  = qdrant.get_collection(TEXT_COLLECTION)
image_info = qdrant.get_collection(IMAGE_COLLECTION)

text_count  = text_info.points_count  if hasattr(text_info,  "points_count") else text_info.vectors_count
image_count = image_info.points_count if hasattr(image_info, "points_count") else image_info.vectors_count

for k, v in {
    "Vector DB"               : "Qdrant (on-disk, in-process)",
    "Text collection"         : TEXT_COLLECTION,
    "Text vectors stored"     : text_count,
    "Text vector dim"         : TEXT_DIM,
    "Image collection"        : IMAGE_COLLECTION,
    "Image vectors stored"    : image_count,
    "Image vector dim"        : IMAGE_DIM,
    "Similarity metric"       : "Cosine",
    "Storage path"            : str(BASE_DIR/"qdrant_storage"),
}.items():
    print(f"  {k:<30}: {v}")

print(f"\n{'='*60}")
print("PHASE 5 — KNOWLEDGE GRAPH SUMMARY")
print(f"{'='*60}")

# Node type distribution
node_types = {}
for _, data in G.nodes(data=True):
    t = data.get("entity_type", "UNKNOWN")
    node_types[t] = node_types.get(t, 0) + 1

# Edge relation distribution
edge_rels = {}
for _, _, data in G.edges(data=True):
    r = data.get("relation", "UNKNOWN")
    edge_rels[r] = edge_rels.get(r, 0) + 1

# Top 10 most connected nodes
degree_map = dict(G.degree())
top_nodes  = sorted(degree_map.items(), key=lambda x: x[1], reverse=True)[:10]

for k, v in {
    "NER model"               : "scispaCy en_ner_bc5cdr_md + lexicon",
    "Total nodes"             : G.number_of_nodes(),
    "Total edges"             : G.number_of_edges(),
    "Graph density"           : f"{nx.density(G):.6f}",
    "Relation strategies"     : "pattern-based + co-occurrence",
}.items():
    print(f"  {k:<30}: {v}")

print(f"\n  Node type distribution:")
for t, count in sorted(node_types.items(), key=lambda x: x[1], reverse=True):
    print(f"    {t:<25}: {count}")

print(f"\n  Edge relation distribution:")
for r, count in sorted(edge_rels.items(), key=lambda x: x[1], reverse=True):
    print(f"    {r:<25}: {count}")

print(f"\n  Top 10 most connected entities:")
for node, degree in top_nodes:
    etype = G.nodes[node].get("entity_type", "?")
    print(f"    [{etype:<15}] {node:<30} degree={degree}")

print(f"\n✓ Phase 4 + 5 complete — proceed to Cell 10 (Hybrid Retrieval)")


In [ ]:
# ============================================================
# CELL 10 — PHASE 6: Hybrid Retrieval System
# ============================================================
print("\n" + "="*60)
print("PHASE 6 — HYBRID RETRIEVAL")
print("="*60)

# Reload text model
print("Loading retrieval models...")
text_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=DEVICE)
reranker   = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

# BM25
print("Building BM25 index...")
def tokenize(text):
    return [t for t in re.findall(r"\b[\w\-\.]+\b", text.lower()) if len(t)>1]

bm25 = BM25Okapi([tokenize(t) for t in df_chunks["text"].tolist()])

# Synonym map
SYNONYMS = {
    "melanoma":             ["cutaneous melanoma","malignant melanoma"],
    "basal cell carcinoma": ["bcc","basal cell cancer"],
    "dermoscopy":           ["dermatoscopy","dermoscope"],
    "cnn":                  ["convolutional neural network"],
    "vit":                  ["vision transformer"],
    "efficientnet":         ["efficientnet-b4","efficientnetb4"],
    "ham10000":             ["ham 10000"],
    "auc":                  ["area under curve","auroc"],
}

def expand_query(q):
    ql = q.lower()
    extra = [syn for term,syns in SYNONYMS.items()
             if term in ql for syn in syns]
    return q + " " + " ".join(extra) if extra else q

def embed_query(q):
    with torch.no_grad():
        return text_model.encode(
            [f"Represent this sentence for retrieval: {q}"],
            normalize_embeddings=True, convert_to_numpy=True
        )[0].astype(np.float32)

def qdrant_search(collection, vector, limit, qfilter=None):
    try:
        return qdrant.query_points(
            collection_name=collection, query=vector.tolist(),
            query_filter=qfilter, limit=limit
        ).points
    except AttributeError:
        return qdrant.search(
            collection_name=collection, query_vector=vector.tolist(),
            query_filter=qfilter, limit=limit
        )

def vector_search(query, top_k=20, source_filter=None):
    q_emb   = embed_query(query)
    qfilter = None
    if source_filter:
        qfilter = Filter(must=[FieldCondition(
            key="source_type", match=MatchValue(value=source_filter)
        )])
    hits = qdrant_search(TEXT_COLLECTION, q_emb, top_k, qfilter)
    return [{"id":h.id,"text":h.payload.get("text",""),
             "score":float(h.score),"section":h.payload.get("section",""),
             "source_type":h.payload.get("source_type",""),
             "pmid":h.payload.get("pmid",""),"pmcid":h.payload.get("pmcid",""),
             "title":h.payload.get("title",""),"method":"vector"} for h in hits]

def bm25_search(query, top_k=20):
    scores  = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_idx:
        if scores[idx] < 0.01: continue
        row = df_chunks.iloc[idx]
        results.append({"id":int(idx),"text":str(row.get("text","")),
                        "score":float(scores[idx]),"section":str(row.get("section","")),
                        "source_type":str(row.get("source_type","")),
                        "pmid":str(row.get("pmid","")),"pmcid":str(row.get("pmcid","")),
                        "title":str(row.get("title","")),"method":"bm25"})
    return results

def graph_search(query, top_k=10):
    q_lower    = query.lower()
    entities   = [n for n in G.nodes() if len(n)>3 and n in q_lower]
    if not entities: return [], ""
    neighbors  = set()
    ctx_lines  = []
    for ent in entities[:3]:
        for _,nbr,data in G.out_edges(ent,data=True):
            neighbors.add(nbr)
            ctx_lines.append(f"{ent} --[{data.get('relation','')}]--> {nbr}")
        for pred,_,data in G.in_edges(ent,data=True):
            neighbors.add(pred)
            ctx_lines.append(f"{pred} --[{data.get('relation','')}]--> {ent}")
    results = []
    seen    = set()
    for nbr in list(neighbors)[:15]:
        mask = df_chunks["text"].str.lower().str.contains(re.escape(nbr),na=False)
        for idx in df_chunks[mask].index[:2]:
            if idx in seen: continue
            seen.add(idx)
            row = df_chunks.iloc[idx]
            results.append({"id":int(idx),"text":str(row.get("text","")),
                            "score":0.7,"section":str(row.get("section","")),
                            "source_type":str(row.get("source_type","")),
                            "pmid":str(row.get("pmid","")),"pmcid":str(row.get("pmcid","")),
                            "title":str(row.get("title","")),"method":"graph",
                            "graph_context":"\n".join(ctx_lines[:10])})
            if len(results)>=top_k: break
    return results, "\n".join(ctx_lines[:10])

def do_rerank(query, candidates, top_k=5):
    if not candidates: return []
    pairs  = [(query, c["text"][:512]) for c in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    for c,s in zip(candidates,scores):
        c["rerank_score"] = float(s)
    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

def hybrid_retrieve(query, top_k=5, source_filter=None):
    expanded     = expand_query(query)
    vec_res      = vector_search(expanded, top_k=20, source_filter=source_filter)
    bm25_res     = bm25_search(expanded, top_k=20)
    graph_res, graph_ctx = graph_search(query, top_k=10)
    # Merge + dedup
    seen, merged = set(), []
    for r in vec_res+bm25_res+graph_res:
        key = r["text"][:80]
        if key not in seen:
            seen.add(key)
            merged.append(r)
    reranked = do_rerank(query, merged, top_k=top_k)
    return {"results":reranked,"graph_ctx":graph_ctx,
            "stats":{"vector":len(vec_res),"bm25":len(bm25_res),
                     "graph":len(graph_res),"merged":len(merged)}}

def retrieve_images(query, top_k=3, label_filter=None):
    q_emb   = embed_query(query)
    qfilter = None
    if label_filter:
        qfilter = Filter(must=[FieldCondition(
            key="label",match=MatchValue(value=label_filter)
        )])
    hits = qdrant_search(IMAGE_COLLECTION, q_emb, top_k, qfilter)
    return [{"id":h.id,"score":float(h.score),
             "label":h.payload.get("label",""),
             "dataset":h.payload.get("dataset",""),
             "image_path":h.payload.get("image_path",""),
             "description":h.payload.get("image_text_description","")}
            for h in hits]

print("Hybrid retrieval ready.")

# Quick smoke test
test = hybrid_retrieve("CNN for melanoma detection dermoscopy", top_k=3)
print(f"Test retrieval: {test['stats']}")
for r in test["results"]:
    print(f"  [{r['method']}] score={r['rerank_score']:.3f} | {r['text'][:80]}...")



In [ ]:
# ============================================================
# CELL 11 — PHASE 7+8+9: Reranking + LLM + Citations
# ============================================================
print("\n" + "="*60)
print("PHASE 7+8+9 — LLM REASONING + CITATIONS")
print("="*60)

# !pip install groq -q
from groq import Groq

GROQ_API_KEY = ""   # ← replace with your Groq key
groq_client  = Groq(api_key=GROQ_API_KEY)

def call_llm_with_retry(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1000,
                temperature=0.1
            )
            return response.choices[0].message.content
        except Exception as e:
            err = str(e)
            if "429" in err and attempt < max_retries - 1:
                print(f"  Rate limited — retrying in 10s...")
                time.sleep(10)
            else:
                return f"LLM error: {e}"
    return "LLM error: max retries exceeded"


RAG_SYSTEM_PROMPT = """You are a medical research-support assistant for skin cancer and dermatology.

IMPORTANT MEDICAL SAFETY RULES:
1. This system is for research and educational support only.
2. You are NOT a doctor and must NOT provide a clinical diagnosis.
3. You must NOT recommend a final treatment plan.
4. You must NOT tell the user they have or do not have cancer.
5. You must advise consulting a qualified dermatologist for diagnosis or treatment decisions.
6. If the user describes urgent symptoms such as rapid growth, bleeding, severe pain, infection, or major change in a lesion, advise urgent medical evaluation.

HALLUCINATION PREVENTION RULES:
1. Answer ONLY using the retrieved context.
2. Every important claim must cite a source using [Source N].
3. If the retrieved context does not contain enough information, say:
   "The retrieved context is insufficient to answer this confidently."
4. Do NOT use outside medical knowledge unless it is explicitly present in the retrieved context.
5. Do NOT invent model names, datasets, metrics, AUC values, treatment details, or clinical claims.
6. If sources disagree, mention the disagreement.
7. Separate research findings from clinical interpretation.
8. Use cautious language: "may", "reported", "associated with", "the retrieved studies suggest".

IMAGE SAFETY RULES:
1. For images, describe retrieved visual similarity only.
2. Do NOT call visual similarity a diagnosis.
3. Do NOT output "100% confidence" for medical labels.
4. Use "similar-image label suggestion" instead of "predicted diagnosis".

Format your response as:
ANSWER: [grounded answer using citations]
MEDICAL SAFETY NOTE: This is not medical advice and not a diagnosis. Please consult a qualified dermatologist.
CONFIDENCE: [HIGH/MEDIUM/LOW based only on retrieved context]
REASONING: [why the answer is supported or uncertain]
CITATIONS: [Source numbers used]
"""


def build_prompt(query, text_context, graph_context, sources):
    source_block = "\n\n".join([
        f"[Source {i+1}] ({s.get('section','')}) — {s.get('title','')[:60]}\n"
        f"PMID:{s.get('pmid','')} | PMCID:{s.get('pmcid','')}\n"
        f"{s.get('text','')}"
        for i, s in enumerate(sources)
    ])
    graph_block = f"\nKNOWLEDGE GRAPH RELATIONSHIPS:\n{graph_context}" \
                  if graph_context else ""
    return f"""{RAG_SYSTEM_PROMPT}

RETRIEVED CONTEXT:
{source_block}
{graph_block}

USER QUESTION: {query}


Provide a cautious research-support answer with citations.
Do not provide a diagnosis.
"""

# CELL 10B — Fix retrieve_images to use CLIP (512-dim) not BGE (1024-dim)
# Run this immediately after Cell 10, before Cell 11

def retrieve_images(query, top_k=3, label_filter=None):
    """Uses CLIP text encoder (512-dim) to match image collection."""
    tokens = mm_tokenizer([query]).to(DEVICE)
    with torch.no_grad():
        q_emb = mm_model.encode_text(tokens)
        q_emb = q_emb / q_emb.norm(dim=-1, keepdim=True)
    q_emb = q_emb.cpu().numpy()[0].astype(np.float32)

    qfilter = None
    if label_filter:
        qfilter = Filter(must=[FieldCondition(
            key="label", match=MatchValue(value=label_filter)
        )])

    hits = qdrant_search(IMAGE_COLLECTION, q_emb, top_k, qfilter)
    return [{"id":          h.id,
             "score":       float(h.score),
             "label":       h.payload.get("label",   ""),
             "dataset":     h.payload.get("dataset", ""),
             "image_path":  h.payload.get("image_path", ""),
             "description": h.payload.get("image_text_description", "")}
            for h in hits]


def retrieve_for_rag(query, top_k=5, include_images=True, image_top_k=3):
    hybrid  = hybrid_retrieve(query, top_k=top_k)
    results = hybrid["results"]

    sources = []
    for i, r in enumerate(results):
        sources.append({
            "rank":         i + 1,
            "pmid":         r.get("pmid",   ""),
            "pmcid":        r.get("pmcid",  ""),
            "title":        r.get("title",  ""),
            "section":      r.get("section",""),
            "method":       r.get("method", ""),
            "rerank_score": r.get("rerank_score", 0.0),
            "text":         r.get("text",   ""),
        })

    # Auto-detect label from query
    label   = None
    q_lower = query.lower()
    for disease in ["melanoma", "basal cell carcinoma",
                    "squamous cell carcinoma", "nevus",
                    "actinic keratosis", "dermatofibroma"]:
        if disease in q_lower:
            label = disease
            break

    images = retrieve_images(query, top_k=image_top_k,
                              label_filter=label) if include_images else []

    return {
        "query":       query,
        "sources":     sources,
        "graph_ctx":   hybrid["graph_ctx"],
        "images":      images,
        "stats":       hybrid["stats"],
    }

print("✓ retrieve_images and retrieve_for_rag ready — proceed to Cell 11")



def ask(query, top_k=5, include_images=True):
    retrieval = retrieve_for_rag(query, top_k=top_k,
                                  include_images=include_images)
    prompt    = build_prompt(
        query,
        text_context  = "\n\n".join([s["text"] for s in retrieval["sources"]]),
        graph_context = retrieval["graph_ctx"],
        sources       = retrieval["sources"]
    )
    answer = call_llm_with_retry(prompt)
    return {
        "query":     query,
        "answer":    answer,
        "sources":   retrieval["sources"],
        "images":    retrieval["images"],
        "graph_ctx": retrieval["graph_ctx"],
        "stats":     retrieval["stats"],
    }


# ── Test the full pipeline ─────────────────────────────────────
print("\nTesting full RAG pipeline...")
test_result = ask(
    "What deep learning models achieve the best accuracy for melanoma detection?",
    top_k=5
)
print(f"\nQuery  : {test_result['query']}")
print(f"Stats  : {test_result['stats']}")
print(f"\nAnswer :\n{test_result['answer']}")
print(f"\nSources: {len(test_result['sources'])}")
for s in test_result["sources"]:
    print(f"  [{s['rank']}] {s['method']} | score={s['rerank_score']:.3f} | "
          f"PMID:{s['pmid']} | {s['title'][:50]}")
print(f"\nImages : {len(test_result['images'])}")
for img in test_result["images"]:
    print(f"  score={img['score']:.4f} | label={img['label']} | {img['dataset']}")


In [ ]:
# ============================================================
# CELL 12 — PHASE 10: Multimodal Image Query chatgpt
# ============================================================

print("\n" + "=" * 60)
print("PHASE 10 — MULTIMODAL IMAGE SIMILARITY SUPPORT")
print("=" * 60)

# mm_model already loaded in Cell 7 — no reload needed
# mm_preprocess = clip_preprocess from Cell 7
mm_preprocess = clip_preprocess

print(f"Using {IMAGE_MODEL_NAME} already in memory.")


MEDICAL_DISCLAIMER = (
    "This system is for research and educational support only. "
    "It is not a clinical diagnostic tool, does not provide medical advice, "
    "and should not replace a dermatologist or qualified medical professional."
)


def embed_image_query(image_path):
    """Embed an image file for visual similarity retrieval."""
    img = Image.open(image_path).convert("RGB")
    tensor = mm_preprocess(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        emb = mm_model.encode_image(tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)

    return emb.cpu().numpy()[0].astype(np.float32)


def multimodal_ask(image_path=None, text_query="", top_k=5):
    """
    Accepts an image + optional text query.

    This function does NOT diagnose the image.
    It retrieves visually similar dermoscopy images and related research context.
    """

    results = {
        "image_path": image_path,
        "text_query": text_query,
        "medical_disclaimer": MEDICAL_DISCLAIMER,
    }

    # ── Step 1: Image similarity search ──────────────────────
    similar_images = []
    similar_label_suggestion = "unknown"

    if image_path and Path(image_path).exists():
        img_emb = embed_image_query(image_path)
        img_hits = qdrant_search(IMAGE_COLLECTION, img_emb, top_k)

        similar_images = [
            {
                "id": h.id,
                "score": float(h.score),
                "label": h.payload.get("label", ""),
                "dataset": h.payload.get("dataset", ""),
                "image_path": h.payload.get("image_path", ""),
            }
            for h in img_hits
        ]

        if similar_images:
            label_counts = Counter(r["label"] for r in similar_images[:3])
            similar_label_suggestion = label_counts.most_common(1)[0][0]

    results["similar_images"] = similar_images
    results["similar_label_suggestion"] = similar_label_suggestion

    # Keep old key also, so Cell 16 does not break
    results["predicted_label"] = similar_label_suggestion

    # ── Step 2: Retrieve related papers ──────────────────────
    if text_query:
        rag_query = text_query
    else:
        rag_query = (
            f"{similar_label_suggestion} dermoscopy visual similarity "
            f"skin lesion research deep learning"
        )

    rag = ask(
        rag_query,
        top_k=top_k,
        include_images=False
    )

    results["text_answer"] = rag["answer"]
    results["sources"] = rag["sources"]
    results["rag_query"] = rag_query

    # ── Step 3: Generate guarded multimodal research summary ──
    visual_similarity = (
        f"{similar_images[0]['score']:.3f}"
        if similar_images else "0.000"
    )

    similar_case_labels = [r["label"] for r in similar_images[:3]]

    research_context = "\n".join([
        f"[Source {i + 1}] PMID:{s.get('pmid', '')} | PMCID:{s.get('pmcid', '')}\n"
        f"Title: {s.get('title', '')}\n"
        f"Text: {s.get('text', '')[:700]}"
        for i, s in enumerate(rag["sources"][:3])
    ])

    mm_prompt = f"""
You are a medical research-support assistant for dermatology and skin cancer literature.

IMPORTANT MEDICAL SAFETY RULES:
1. This system is for research and educational support only.
2. You are NOT a doctor and must NOT provide a clinical diagnosis.
3. You must NOT tell the user they have or do not have cancer.
4. You must NOT recommend a final treatment plan.
5. You must advise consulting a qualified dermatologist for diagnosis or treatment decisions.
6. If the image or question suggests urgent symptoms such as rapid growth, bleeding, severe pain, infection, or major change in a lesion, advise urgent medical evaluation.

HALLUCINATION PREVENTION RULES:
1. Use ONLY the retrieved research context below.
2. Every important claim must cite a source using [Source N].
3. If the retrieved context does not contain enough information, say:
   "The retrieved context is insufficient to answer this confidently."
4. Do NOT invent visual features, datasets, model scores, treatment details, or clinical recommendations.
5. Do NOT use outside medical knowledge unless it appears in the retrieved context.
6. Do NOT output "100% confidence" for a medical label.
7. Separate visual similarity from research findings.

IMAGE SAFETY RULES:
1. The image result is visual similarity support only.
2. Do NOT call it a diagnosis.
3. Use "similar-image label suggestion", not "predicted diagnosis".
4. Visual similarity score is not medical confidence.

MEDICAL DISCLAIMER:
{MEDICAL_DISCLAIMER}

Image Similarity Result:
- Similar-image label suggestion: {similar_label_suggestion}
- Top similar case labels: {similar_case_labels}
- Top visual similarity score: {visual_similarity}

Retrieved Research Context:
{research_context}

User text query:
{text_query}

Write the response in this format:

ANSWER:
[Brief research-support answer grounded only in the retrieved context.]

IMAGE SIMILARITY NOTE:
[Explain that the label is based on retrieved visually similar images, not diagnosis.]

RESEARCH FINDINGS:
[Summarize only findings supported by retrieved sources with citations.]

SAFETY NOTE:
This is not medical advice and not a diagnosis. A dermatologist should evaluate any concerning lesion.

CONFIDENCE:
[HIGH / MEDIUM / LOW based only on retrieved context quality.]

CITATIONS:
[List Source numbers used.]
"""

    results["mm_answer"] = call_llm_with_retry(mm_prompt)

    return results


# ── Test with a sample HAM10000 image ─────────────────────────

HAM_DIR = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")
sample_images = list((HAM_DIR / "HAM10000_images_part_1").glob("*.jpg"))

if sample_images:
    sample_path = str(sample_images[0])

    print(f"Testing multimodal similarity support with: {sample_path}")

    mm_result = multimodal_ask(
        image_path=sample_path,
        text_query="What research context is related to visually similar skin lesions?",
        top_k=3
    )

    print(f"\nSimilar-image label suggestion : {mm_result['similar_label_suggestion']}")
    print("Note: This is based on visual similarity, not clinical diagnosis.")

    print(f"\nSimilar images retrieved        : {len(mm_result['similar_images'])}")
    print(f"RAG query used                  : {mm_result['rag_query']}")

    print(f"\nTop similar images:")
    for img in mm_result["similar_images"]:
        print(
            f"  score={img['score']:.4f} | "
            f"label={img['label']} | dataset={img['dataset']}"
        )

    print(f"\nMultimodal Research-Support Answer:\n{mm_result['mm_answer'][:900]}...")

    print(f"\nSources used:")
    for s in mm_result["sources"][:3]:
        print(f"  [{s['rank']}] PMID:{s['pmid']} | {s['title'][:60]}")

else:
    print("No sample images found — check HAM10000 dataset path")


In [ ]:
# Run this first, then re-run only Part C

import subprocess
subprocess.run(["pip", "install", "langchain-groq", "langchain-huggingface",
                "ragas", "datasets", "-q"], check=True)
print("Done — now re-run Part C only")


In [ ]:
# ============================================================
# CELL 13 — PHASE 11: Complete Evaluation
# ============================================================
print("\n" + "="*60)
print("PHASE 11 — EVALUATION")
print("="*60)

import subprocess
subprocess.run(["pip", "install", "langchain-groq", "langchain-huggingface",
                "ragas", "datasets", "-q"], check=True)

import time
import ast
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithoutReference,
    LLMContextRecall,
)
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

EVAL_DIR = BASE_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TEST_QUERIES = [
    {
        "query": "What CNN architectures are used for melanoma classification?",
        "ground_truth": "CNN architectures such as ResNet, EfficientNet, VGG, DenseNet, and Inception are used for melanoma classification from dermoscopy images."
    },
    {
        "query": "What is the accuracy of deep learning for skin cancer detection?",
        "ground_truth": "Deep learning models report high accuracy for skin cancer detection, varying by dataset, lesion type, and validation method."
    },
    {
        "query": "How is dermoscopy used in skin lesion diagnosis?",
        "ground_truth": "Dermoscopy is a non-invasive imaging technique used to visualize subsurface skin structures and support assessment of skin lesions."
    },
    {
        "query": "What datasets are commonly used for skin cancer classification?",
        "ground_truth": "Common datasets include HAM10000, ISIC Archive, PH2, and Derm7pt."
    },
    {
        "query": "What is the difference between melanoma and nevus in dermoscopy?",
        "ground_truth": "Melanoma shows asymmetric structure, irregular borders, multiple colors, while benign nevi are more symmetric and regular."
    },
]


# ============================================================
# PART A — Offline Retrieval Metrics (no LLM, runs instantly)
# ============================================================
print("\n── PART A: Offline Retrieval Metrics ────────────────────")

offline_results = []

for item in tqdm(TEST_QUERIES, desc="Retrieval eval"):
    query  = item["query"]
    hybrid = hybrid_retrieve(query, top_k=5)
    results= hybrid["results"]

    rerank_scores   = [r.get("rerank_score", 0) for r in results]
    methods_used    = set(r.get("method","")    for r in results)
    sources_with_id = [r for r in results if r.get("pmid","") or r.get("pmcid","")]
    has_text        = [r for r in results if len(r.get("text","")) > 50]

    query_keywords  = [w for w in query.lower().split()
                       if len(w) > 4 and w not in
                       {"what","which","where","when","does","used","skin"}]
    top_text        = results[0].get("text","").lower() if results else ""
    keyword_hits    = sum(1 for kw in query_keywords if kw in top_text)
    keyword_rate    = round(keyword_hits / max(len(query_keywords), 1), 3)

    offline_results.append({
        "query":               query[:60],
        "top_rerank_score":    round(max(rerank_scores), 3) if rerank_scores else 0,
        "avg_rerank_score":    round(sum(rerank_scores)/max(len(rerank_scores),1), 3),
        "citation_coverage":   round(len(sources_with_id)/max(len(results),1), 3),
        "context_coverage":    round(len(has_text)/max(len(results),1), 3),
        "keyword_hit_rate":    keyword_rate,
        "retrieval_diversity": len(methods_used),
        "methods_used":        "+".join(sorted(methods_used)),
        "has_graph_context":   bool(hybrid.get("graph_ctx","")),
        "vector_hits":         hybrid["stats"]["vector"],
        "bm25_hits":           hybrid["stats"]["bm25"],
        "graph_hits":          hybrid["stats"]["graph"],
        "total_candidates":    hybrid["stats"]["merged"],
    })

df_offline = pd.DataFrame(offline_results)
df_offline.to_csv(EVAL_DIR / "offline_eval.csv", index=False)

print(f"\n{'='*55}")
print("OFFLINE RETRIEVAL EVALUATION SUMMARY")
print(f"{'='*55}")
print(f"  Queries evaluated       : {len(df_offline)}")
print(f"  Avg top rerank score    : {df_offline['top_rerank_score'].mean():.3f}")
print(f"  Avg mean rerank score   : {df_offline['avg_rerank_score'].mean():.3f}")
print(f"  Avg citation coverage   : {df_offline['citation_coverage'].mean():.3f}")
print(f"  Avg context coverage    : {df_offline['context_coverage'].mean():.3f}")
print(f"  Avg keyword hit rate    : {df_offline['keyword_hit_rate'].mean():.3f}")
print(f"  Avg retrieval diversity : {df_offline['retrieval_diversity'].mean():.1f} methods")
print(f"  Graph context used      : {df_offline['has_graph_context'].sum()}/{len(df_offline)} queries")

print(f"\nPer-query breakdown:")
for _, row in df_offline.iterrows():
    print(f"\n  [{row['query'][:55]}]")
    print(f"    Top score   : {row['top_rerank_score']}")
    print(f"    Avg score   : {row['avg_rerank_score']}")
    print(f"    Keyword hit : {row['keyword_hit_rate']}")
    print(f"    Citations   : {row['citation_coverage']}")
    print(f"    Methods     : {row['methods_used']}")
    print(f"    Graph ctx   : {row['has_graph_context']}")
    print(f"    Candidates  : {row['total_candidates']} → reranked to 5")


# ============================================================
# PART B — Generate LLM Answers
# ============================================================
print(f"\n── PART B: Generating LLM Answers ───────────────────────")

ragas_rows = []

for item in tqdm(TEST_QUERIES, desc="Generating answers"):
    answer = ""
    for attempt in range(3):
        result = ask(item["query"], top_k=5, include_images=False)
        answer = result.get("answer", "")
        if "LLM error" not in answer and len(answer) > 50:
            break
        wait = 20 * (attempt + 1)
        print(f"  Rate limited — waiting {wait}s (attempt {attempt+1}/3)...")
        time.sleep(wait)

    if "LLM error" in answer or len(answer) < 50:
        print(f"  ✗ Skipped: {item['query'][:50]}")
        continue

    contexts = [s.get("text","") for s in result.get("sources",[])
                if s.get("text","")]
    ragas_rows.append({
        "user_input":         item["query"],
        "response":           answer,
        "retrieved_contexts": contexts,
        "reference":          item["ground_truth"],
    })
    time.sleep(8)

print(f"\n  Generated {len(ragas_rows)}/{len(TEST_QUERIES)} answers")


# ============================================================
# PART C — RAGAS Metrics
# ============================================================
if len(ragas_rows) == 0:
    print("\n── PART C: RAGAS Skipped (Groq quota exceeded) ─────────")
    print("  Re-run after 1 PM PKT to get RAGAS metrics.")
    print("  Offline metrics above are already saved.")

else:
    print(f"\n── PART C: RAGAS Metrics on {len(ragas_rows)} answers ──────────")
    try:
        ragas_llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0,
            api_key=GROQ_API_KEY
        )
        ragas_embeddings = HuggingFaceEmbeddings(
            model_name="BAAI/bge-large-en-v1.5",
            model_kwargs={"device": DEVICE},
            encode_kwargs={"normalize_embeddings": True}
        )

        df_ragas_input = pd.DataFrame(ragas_rows)
        df_ragas_input.to_csv(EVAL_DIR / "ragas_input.csv", index=False)
        ragas_dataset  = Dataset.from_pandas(df_ragas_input)

        print("Running RAGAS metrics...")
        ragas_result = evaluate(
            dataset    = ragas_dataset,
            metrics    = [
                Faithfulness(),
                AnswerRelevancy(),
                LLMContextPrecisionWithoutReference(),
                LLMContextRecall(),
            ],
            llm        = ragas_llm,
            embeddings = ragas_embeddings,
        )

        df_ragas = ragas_result.to_pandas()
        df_ragas.to_csv(EVAL_DIR / "ragas_results.csv", index=False)

        print(f"\n{'='*55}")
        print("RAGAS EVALUATION SUMMARY")
        print(f"{'='*55}")
        for col, label in [
            ("faithfulness",                            "Faithfulness       "),
            ("answer_relevancy",                        "Answer Relevancy   "),
            ("llm_context_precision_without_reference", "Context Precision  "),
            ("context_recall",                          "Context Recall     "),
        ]:
            val    = df_ragas[col].mean() if col in df_ragas.columns else float("nan")
            status = "✓" if not pd.isna(val) else "✗"
            print(f"  {status} {label}: {val:.3f}" if not pd.isna(val)
                  else f"  ✗ {label}: N/A")

        print(f"\n{'='*55}")
        print("COMBINED EVALUATION SUMMARY")
        print(f"{'='*55}")
        print(f"  [Retrieval]  Avg rerank score    : {df_offline['avg_rerank_score'].mean():.3f}")
        print(f"  [Retrieval]  Citation coverage   : {df_offline['citation_coverage'].mean():.3f}")
        print(f"  [Retrieval]  Keyword hit rate    : {df_offline['keyword_hit_rate'].mean():.3f}")
        print(f"  [Retrieval]  Diversity           : {df_offline['retrieval_diversity'].mean():.1f} methods")
        if "faithfulness" in df_ragas.columns:
            print(f"  [RAGAS]      Faithfulness        : {df_ragas['faithfulness'].mean():.3f}")
            print(f"  [RAGAS]      Answer Relevancy    : {df_ragas['answer_relevancy'].mean():.3f}")
            print(f"  [RAGAS]      Context Precision   : {df_ragas['llm_context_precision_without_reference'].mean():.3f}")
            print(f"  [RAGAS]      Context Recall      : {df_ragas['context_recall'].mean():.3f}")

        display(df_ragas[["user_input", "faithfulness",
                           "answer_relevancy", "context_recall"]])

    except Exception as e:
        print(f"  RAGAS error: {e}")
        print("  Offline metrics are saved.")


# ── Final save ─────────────────────────────────────────────────
print(f"\nSaved files:")
print(f"  {EVAL_DIR}/offline_eval.csv")
if (EVAL_DIR / "ragas_results.csv").exists():
    print(f"  {EVAL_DIR}/ragas_results.csv")

display(df_offline)


In [ ]:
# ============================================================
# CELL 13B — PHASE 11B: Retrieval Strategy Comparison
# BM25 only vs Dense only vs Hybrid
# ============================================================
print("\n" + "="*60)
print("PHASE 11B — RETRIEVAL STRATEGY COMPARISON")
print("="*60)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('inline')
%matplotlib inline

TEST_QUERIES_COMPARE = [
    "CNN architecture melanoma classification dermoscopy",
    "basal cell carcinoma deep learning accuracy",
    "dermoscopy skin lesion diagnosis transformer",
    "HAM10000 ISIC dataset skin cancer classification",
    "melanoma nevus difference visual features",
]

comparison_results = []

for query in tqdm(TEST_QUERIES_COMPARE, desc="Comparing strategies"):

    # ── Strategy 1: BM25 only ─────────────────────────────────
    bm25_only = bm25_search(query, top_k=5)
    bm25_scores = [r.get("rerank_score", 0) for r in
                   do_rerank(query, bm25_only, top_k=5)]

    # ── Strategy 2: Dense (vector) only ──────────────────────
    dense_only = vector_search(query, top_k=5)
    dense_scores = [r.get("rerank_score", 0) for r in
                    do_rerank(query, dense_only, top_k=5)]

    # ── Strategy 3: Hybrid (vector + BM25 + graph) ───────────
    hybrid = hybrid_retrieve(query, top_k=5)
    hybrid_scores = [r.get("rerank_score", 0) for r in hybrid["results"]]

    comparison_results.append({
        "query":              query[:45],
        # Top-1 scores
        "bm25_top1":          round(max(bm25_scores),  3) if bm25_scores  else 0,
        "dense_top1":         round(max(dense_scores), 3) if dense_scores else 0,
        "hybrid_top1":        round(max(hybrid_scores),3) if hybrid_scores else 0,
        # Mean scores
        "bm25_mean":          round(sum(bm25_scores)/max(len(bm25_scores),1),   3),
        "dense_mean":         round(sum(dense_scores)/max(len(dense_scores),1),  3),
        "hybrid_mean":        round(sum(hybrid_scores)/max(len(hybrid_scores),1),3),
        # Candidate pool sizes
        "bm25_candidates":    len(bm25_only),
        "dense_candidates":   len(dense_only),
        "hybrid_candidates":  hybrid["stats"]["merged"],
        # Unique results (dedup check)
        "bm25_unique_pmids":  len(set(r.get("pmid","") for r in bm25_only  if r.get("pmid",""))),
        "dense_unique_pmids": len(set(r.get("pmid","") for r in dense_only if r.get("pmid",""))),
        "hybrid_graph_used":  bool(hybrid.get("graph_ctx","")),
    })

df_compare = pd.DataFrame(comparison_results)
df_compare.to_csv(EVAL_DIR / "retrieval_comparison.csv", index=False)

# ── Print table ───────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"{'Query':<45} {'BM25':>7} {'Dense':>7} {'Hybrid':>7}")
print(f"{'─'*45} {'─'*7} {'─'*7} {'─'*7}")
for _, row in df_compare.iterrows():
    winner = max([
        ("BM25",   row["hybrid_top1"] if row["hybrid_top1"] else 0),
        ("Dense",  row["dense_top1"]),
        ("Hybrid", row["hybrid_top1"]),
    ], key=lambda x: x[1])[0]
    print(f"{row['query']:<45} "
          f"{row['bm25_top1']:>7.3f} "
          f"{row['dense_top1']:>7.3f} "
          f"{row['hybrid_top1']:>7.3f}")

print(f"\n{'─'*70}")
print(f"{'AVERAGE':<45} "
      f"{df_compare['bm25_top1'].mean():>7.3f} "
      f"{df_compare['dense_top1'].mean():>7.3f} "
      f"{df_compare['hybrid_top1'].mean():>7.3f}")

print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(f"  BM25 only    avg top-1 : {df_compare['bm25_top1'].mean():.3f} | "
      f"avg mean : {df_compare['bm25_mean'].mean():.3f}")
print(f"  Dense only   avg top-1 : {df_compare['dense_top1'].mean():.3f} | "
      f"avg mean : {df_compare['dense_mean'].mean():.3f}")
print(f"  Hybrid       avg top-1 : {df_compare['hybrid_top1'].mean():.3f} | "
      f"avg mean : {df_compare['hybrid_mean'].mean():.3f}")

# Determine winner
scores = {
    "BM25":   df_compare["bm25_top1"].mean(),
    "Dense":  df_compare["dense_top1"].mean(),
    "Hybrid": df_compare["hybrid_top1"].mean(),
}
winner = max(scores, key=scores.get)
print(f"\n  Best strategy: {winner} "
      f"(avg top-1 score = {scores[winner]:.3f})")


# ── Plot comparison ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Top-1 rerank score per query
x     = range(len(df_compare))
width = 0.25

axes[0].bar([i - width for i in x], df_compare["bm25_top1"],
            width, label="BM25 only",    color="#4A90D9", alpha=0.85)
axes[0].bar([i         for i in x], df_compare["dense_top1"],
            width, label="Dense only",   color="#E67E22", alpha=0.85)
axes[0].bar([i + width for i in x], df_compare["hybrid_top1"],
            width, label="Hybrid",       color="#27AE60", alpha=0.85)

axes[0].set_title("Top-1 Rerank Score by Strategy",
                  fontsize=13, fontweight="bold")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(
    [q[:20]+"..." for q in df_compare["query"]],
    rotation=30, ha="right", fontsize=8
)
axes[0].set_ylabel("Rerank Score")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Plot 2: Average scores bar chart
strategies = ["BM25 only", "Dense only", "Hybrid"]
avg_top1   = [df_compare["bm25_top1"].mean(),
              df_compare["dense_top1"].mean(),
              df_compare["hybrid_top1"].mean()]
avg_mean   = [df_compare["bm25_mean"].mean(),
              df_compare["dense_mean"].mean(),
              df_compare["hybrid_mean"].mean()]
colors     = ["#4A90D9", "#E67E22", "#27AE60"]

bars = axes[1].bar(strategies, avg_top1, color=colors, alpha=0.85, width=0.4)
axes[1].set_title("Average Top-1 Score Across All Queries",
                  fontsize=13, fontweight="bold")
axes[1].set_ylabel("Avg Top-1 Rerank Score")
axes[1].grid(axis="y", alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, avg_top1):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.1,
                 f"{val:.3f}", ha="center", va="bottom",
                 fontweight="bold", fontsize=11)

# Highlight winner
best_idx = avg_top1.index(max(avg_top1))
bars[best_idx].set_edgecolor("black")
bars[best_idx].set_linewidth(2.5)
axes[1].text(best_idx, avg_top1[best_idx] + 0.3,
             "★ BEST", ha="center", fontsize=10,
             color="black", fontweight="bold")

plt.suptitle("Retrieval Strategy Comparison: BM25 vs Dense vs Hybrid",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nSaved: {EVAL_DIR}/retrieval_comparison.csv")
display(df_compare[["query","bm25_top1","dense_top1","hybrid_top1",
                     "hybrid_candidates","hybrid_graph_used"]])


In [ ]:
# ============================================================
# CELL 14 — PHASE 12: FastAPI Deployment Code
# ============================================================
print("\n" + "="*60)
print("PHASE 12 — DEPLOYMENT CODE")
print("="*60)

deploy_dir = BASE_DIR / "deployment"
deploy_dir.mkdir(exist_ok=True)

# ── main.py ───────────────────────────────────────────────────
fastapi_code = '''
# ============================================================
# main.py — FastAPI Backend for Skin Cancer Medical RAG
# Deploy locally : uvicorn main:app --host 0.0.0.0 --port 8000
# Deploy Docker  : docker build -t skin-rag . && docker run -p 8000:8000 skin-rag
# ============================================================

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uvicorn, tempfile
from pathlib import Path

app = FastAPI(
    title       = "Skin Cancer Medical RAG API",
    description = "Multimodal RAG system for skin cancer research support",
    version     = "1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins  = ["*"],
    allow_methods  = ["*"],
    allow_headers  = ["*"],
)

class TextQuery(BaseModel):
    query:          str
    top_k:          int  = 5
    include_images: bool = True
    source_filter:  Optional[str] = None

class RAGResponse(BaseModel):
    query:   str
    answer:  str
    sources: list
    images:  list
    stats:   dict

# ── Health check ──────────────────────────────────────────────
@app.get("/health")
def health():
    return {
        "status":  "ok",
        "llm":     "Groq Llama-3.3-70B",
        "embedder":"BAAI/bge-large-en-v1.5",
        "images":  "BiomedCLIP",
        "vector_db": "Qdrant",
    }

# ── Text query endpoint ───────────────────────────────────────
@app.post("/ask", response_model=RAGResponse)
def ask_endpoint(request: TextQuery):
    """
    Text query → grounded medical research answer with citations.
    NOT a diagnostic tool. For research and education only.
    """
    try:
        result = ask(
            request.query,
            top_k          = request.top_k,
            include_images = request.include_images
        )
        return RAGResponse(
            query  = result["query"],
            answer = result["answer"],
            sources= [{
                "rank":    s["rank"],
                "pmid":    s["pmid"],
                "pmcid":   s["pmcid"],
                "title":   s["title"],
                "section": s["section"],
                "score":   s["rerank_score"],
                "method":  s["method"],
                "snippet": s["text"][:300],
            } for s in result["sources"]],
            images = result["images"],
            stats  = result["stats"]
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ── Image upload endpoint ─────────────────────────────────────
@app.post("/ask-image")
async def ask_image_endpoint(
    file:       UploadFile = File(...),
    text_query: str        = ""
):
    """
    Upload a dermoscopy image → visual similarity search +
    related research retrieval + research-support summary.
    NOT a diagnostic tool.
    """
    try:
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
            tmp.write(await file.read())
            tmp_path = tmp.name

        result = multimodal_ask(
            image_path = tmp_path,
            text_query = text_query,
            top_k      = 3
        )
        Path(tmp_path).unlink(missing_ok=True)

        return {
            "similar_label_suggestion": result.get("similar_label_suggestion",""),
            "similar_images":           result["similar_images"][:3],
            "mm_answer":                result["mm_answer"],
            "sources":                  result["sources"][:3],
            "medical_disclaimer":       result.get("medical_disclaimer",""),
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ── Stats endpoint ────────────────────────────────────────────
@app.get("/stats")
def stats():
    return {
        "text_chunks":    len(df_chunks),
        "images_indexed": len(df_images),
        "kg_nodes":       G.number_of_nodes(),
        "kg_edges":       G.number_of_edges(),
    }

# ── Retrieve only (no LLM) ────────────────────────────────────
@app.get("/retrieve")
def retrieve_endpoint(query: str, top_k: int = 5):
    """Returns raw retrieved chunks without LLM answer generation."""
    try:
        result = retrieve_for_rag(query, top_k=top_k, include_images=True)
        return {
            "query":   query,
            "sources": result["sources"],
            "images":  result["images"],
            "stats":   result["stats"],
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# ── Dockerfile ────────────────────────────────────────────────
dockerfile = '''
FROM python:3.11-slim

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    git curl build-essential \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
'''

# ── requirements.txt ─────────────────────────────────────────
requirements = """\
fastapi==0.110.0
uvicorn==0.29.0
python-multipart==0.0.9
sentence-transformers==2.7.0
open-clip-torch==2.24.0
qdrant-client==1.9.0
rank-bm25==0.2.2
networkx==3.3
groq==0.9.0
Pillow==10.3.0
numpy==1.26.4
pandas==2.2.2
torch==2.3.0
biopython==1.83
beautifulsoup4==4.12.3
lxml==5.2.1
scispacy==0.5.4
"""

# ── docker-compose.yml ────────────────────────────────────────
docker_compose = """\
version: "3.8"
services:
  skin-rag-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - GROQ_API_KEY=${GROQ_API_KEY}
    volumes:
      - ./data:/app/data
    restart: unless-stopped
"""

# ── .env template ─────────────────────────────────────────────
env_template = """\
# Environment variables for Skin Cancer RAG API
GROQ_API_KEY=your_groq_api_key_here
"""

# ── API usage README ──────────────────────────────────────────
readme = """\
# Skin Cancer Medical RAG — API

## Quick Start

### Local
```bash
pip install -r requirements.txt
uvicorn main:app --host 0.0.0.0 --port 8000
```

### Docker
```bash
docker build -t skin-rag .
docker run -p 8000:8000 -e GROQ_API_KEY=your_key skin-rag
```

## Endpoints

| Method | Endpoint      | Description                          |
|--------|---------------|--------------------------------------|
| GET    | /health       | System status                        |
| GET    | /stats        | Database statistics                  |
| POST   | /ask          | Text query → answer + citations      |
| POST   | /ask-image    | Image upload → similarity + research |
| GET    | /retrieve     | Raw retrieval without LLM            |

## Example Usage

### Text Query
```bash
curl -X POST http://localhost:8000/ask \\
  -H "Content-Type: application/json" \\
  -d '{"query": "What is the best model for melanoma detection?", "top_k": 5}'
```

### Image Query
```bash
curl -X POST http://localhost:8000/ask-image \\
  -F "file=@/path/to/dermoscopy.jpg" \\
  -F "text_query=What is this skin lesion?"
```

## Disclaimer
This system is for research and educational support only.
It is NOT a clinical diagnostic tool and must not replace
a qualified dermatologist or medical professional.
"""

# ── Write all files ───────────────────────────────────────────
(deploy_dir / "main.py").write_text(fastapi_code)
(deploy_dir / "Dockerfile").write_text(dockerfile)
(deploy_dir / "requirements.txt").write_text(requirements)
(deploy_dir / "docker-compose.yml").write_text(docker_compose)
(deploy_dir / ".env.template").write_text(env_template)
(deploy_dir / "README.md").write_text(readme)

print("Deployment files written to data/deployment/")
print(f"  main.py            — FastAPI backend (5 endpoints)")
print(f"  Dockerfile         — Container definition")
print(f"  docker-compose.yml — Compose configuration")
print(f"  requirements.txt   — Python dependencies")
print(f"  .env.template      — Environment variables template")
print(f"  README.md          — API usage guide")
print(f"\nEndpoints:")
print(f"  GET  /health       — System status")
print(f"  GET  /stats        — Database stats")
print(f"  POST /ask          — Text query → answer + citations")
print(f"  POST /ask-image    — Image → similarity + research")
print(f"  GET  /retrieve     — Raw retrieval without LLM")
print(f"\nTo deploy locally:")
print(f"  cd {deploy_dir}")
print(f"  pip install -r requirements.txt")
print(f"  uvicorn main:app --host 0.0.0.0 --port 8000")


In [ ]:
# ============================================================
# CELL 15 — Final Summary
# ============================================================
print("\n" + "="*60)
print("COMPLETE PIPELINE SUMMARY")
print("="*60)

summary = {
    "system":        "Multimodal Medical RAG — Skin Cancer",
    "timestamp":     datetime.now().isoformat(),
    "phases_completed": list(range(1,13)),
    "phase_1": {
        "pubmed_papers":    len(paper_ids),
        "abstracts":        len(df_abstracts),
        "pmc_linked":       len(df_pmc),
        "fulltext_downloaded": len(df_downloaded),
        "images_indexed":   len(df_images),
    },
    "phase_2": {
        "text_chunks":      len(df_chunks),
        "chunk_sources":    df_chunks["source_type"].value_counts().to_dict(),
        "avg_words":        round(float(df_chunks["word_count"].mean()),1),
    },
    "phase_3": {
        "text_model":       "BAAI/bge-large-en-v1.5",
        "image_model":      IMAGE_MODEL_NAME,
        "text_embed_dim":   TEXT_DIM,
        "image_embed_dim":  IMAGE_DIM,
        "text_embeddings":  text_embeddings.shape[0],
        "image_embeddings": image_embeddings.shape[0],
    },
    "phase_4": {
        "vector_db":        "Qdrant (on-disk)",
        "text_vectors":     len(df_chunks),
        "image_vectors":    len(df_images),
    },
    "phase_5": {
        "kg_nodes":         G.number_of_nodes(),
        "kg_edges":         G.number_of_edges(),
        "entity_types":     list(ENTITY_LEXICON.keys()),
    },
    "phase_6": {
        "retrieval_methods": ["vector","bm25","graph"],
        "reranker":         "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "query_expansion":  True,
    },
    "phase_7_8_9": {
    "llm": "Llama-3.3-70B via Groq API",
    "citation_format": "[Source N] with PMID/PMCID",
    "confidence_levels": ["HIGH", "MEDIUM", "LOW"],
    },
    "phase_10": {
    "multimodal":       True,
    "image_query":      True,
    "visual_similarity_support": True,
    "clinical_diagnosis": False,
    },
    "phase_11": {
    "eval_queries": len(df_ragas) if "df_ragas" in globals() else 0,
    "avg_faithfulness": round(
        float(df_ragas["faithfulness"].mean()), 3
    ) if "df_ragas" in globals() and "faithfulness" in df_ragas.columns else None,

    "avg_answer_relevancy": round(
        float(df_ragas["answer_relevancy"].mean()), 3
    ) if "df_ragas" in globals() and "answer_relevancy" in df_ragas.columns else None,

    "avg_context_precision": round(
        float(df_ragas["llm_context_precision_without_reference"].mean()), 3
    ) if "df_ragas" in globals() and
         "llm_context_precision_without_reference" in df_ragas.columns else None,

    "avg_context_recall": round(
        float(df_ragas["context_recall"].mean()), 3
    ) if "df_ragas" in globals() and "context_recall" in df_ragas.columns else None,
    },
    "phase_12": {
        "backend":          "FastAPI",
        "endpoints":        ["/health","/ask","/ask-image","/stats"],
        "container":        "Docker",
    },
    "status": "All phases complete."
}

with open(BASE_DIR/"pipeline_summary.json","w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\n✓ Multimodal Medical RAG research-support pipeline complete.")


In [ ]:
# ============================================================
# HOW TO USE — Interactive Query Cell
# ============================================================
# Run this cell anytime to ask medical research-support questions

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

MEDICAL_DISCLAIMER = (
    "This system is for research and educational support only. "
    "It is not a clinical diagnostic tool and should not replace "
    "a dermatologist or medical professional."
)


def show_retrieved_images(images, title="RETRIEVED IMAGES"):
    """
    Display retrieved similar images with label, dataset, score, and path.
    """

    if not images:
        print(f"\n{title}: No images found.")
        return

    print(f"\n{title} ({len(images)}):")

    for i, img in enumerate(images, 1):
        score = img.get("score", None)
        label = img.get("label", "unknown")
        dataset = img.get("dataset", "unknown")
        image_path = img.get("image_path", None)

        if score is not None:
            score_text = f"{score:.4f}"
        else:
            score_text = "N/A"

        print(
            f"\n[{i}] similarity_score={score_text} | "
            f"label={label} | dataset={dataset}"
        )

        if image_path:
            print(f"    path={image_path}")

        try:
            if image_path is None:
                print("    Image path missing.")
                continue

            image_path = Path(image_path)

            if not image_path.exists():
                print("    Image file not found at this path.")
                continue

            image = Image.open(image_path)

            plt.figure(figsize=(4, 4))
            plt.imshow(image)
            plt.axis("off")
            plt.title(
                f"{label} | {dataset} | score={score_text}",
                fontsize=10
            )
            plt.show()

        except Exception as e:
            print(f"    Could not display image: {e}")


def medical_rag_query(question, include_images=True):
    """
    Ask any skin cancer / dermatology research question.
    Returns grounded answer with citations and relevant images.
    """

    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print(f"{'='*60}")

    print(f"\nMEDICAL DISCLAIMER:\n{MEDICAL_DISCLAIMER}")

    result = ask(question, top_k=5, include_images=include_images)

    print(f"\nANSWER:\n{result['answer']}")

    print(f"\nSOURCES ({len(result['sources'])}):")
    for s in result["sources"]:
        print(f"  [{s['rank']}] PMID:{s['pmid']} | {s['title'][:65]}")
        print(f"       score={s['rerank_score']:.3f} | method={s['method']}")

    if result.get("graph_ctx"):
        print(f"\nGRAPH CONTEXT:\n{result['graph_ctx'][:300]}")

    if result.get("images"):
        show_retrieved_images(
            result["images"],
            title="RELEVANT SIMILAR IMAGES"
        )

    print(f"\nRETRIEVAL STATS: {result['stats']}")

    return result


def image_similarity_support(image_path, question=""):
    """
    Upload a dermoscopy image → get visually similar images
    and related research context.

    This is NOT a medical diagnosis function.
    """

    print(f"\n{'='*60}")
    print(f"IMAGE SIMILARITY SUPPORT: {Path(image_path).name}")
    print(f"{'='*60}")

    print(f"\nMEDICAL DISCLAIMER:\n{MEDICAL_DISCLAIMER}")

    print("\nQUERY IMAGE:")
    try:
        query_image = Image.open(image_path)

        plt.figure(figsize=(4, 4))
        plt.imshow(query_image)
        plt.axis("off")
        plt.title("Query Image", fontsize=10)
        plt.show()

    except Exception as e:
        print(f"Could not display query image: {e}")

    result = multimodal_ask(
        image_path=image_path,
        text_query=question,
        top_k=3
    )

    print(f"\nSIMILAR-IMAGE LABEL SUGGESTION: {result['predicted_label']}")
    print("Note: This label is based on visual similarity, not clinical diagnosis.")

    show_retrieved_images(
        result["similar_images"],
        title="TOP SIMILAR IMAGES"
    )

    print(f"\nRESEARCH-SUPPORT ANSWER:\n{result['mm_answer']}")

    print(f"\nRELATED PAPERS:")
    for s in result["sources"][:3]:
        print(f"  PMID:{s['pmid']} | {s['title'][:65]}")

    return result


# ============================================================
# EXAMPLE QUERIES — run these or write your own
# ============================================================

medical_rag_query(
    "What is the difference between melanoma and nevus in dermoscopy?"
)

medical_rag_query(
    "Which deep learning model has the highest AUC for basal cell carcinoma detection?"
)

medical_rag_query(
    "What datasets are used to train skin cancer classification models?"
)

# Image similarity support example
sample = str(list((HAM_DIR / "HAM10000_images_part_1").glob("*.jpg"))[5])

image_similarity_support(
    sample,
    question="What research context is related to visually similar skin lesions?"
)
